# Практическое занятие 10
## Методы расстояний для выявления аномалий в кибербезопасности

Продолжительность: 80 минут. Столбцы `scenario` и `is_attack` используются только после получения прогнозов для учебной проверки качества.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.preprocessing import StandardScaler
%config InlineBackend.figure_format = 'retina'

In [ ]:
FEATURES = [
    'connections_per_min',
    'unique_destinations',
    'outbound_mb'
]

data_path = Path('cyber_distance_anomalies.csv')
df = pd.read_csv(data_path)

reference_df = df.loc[df['dataset_part'] == 'reference_normal'].copy()
monitoring_df = df.loc[df['dataset_part'] == 'monitoring'].copy()

print('Вся выборка:', df.shape)
print('Профиль нормы:', reference_df.shape)
print('Мониторинг:', monitoring_df.shape)

Вспомогательные функции.

In [ ]:
def print_detection_result(method_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(method_name)
    print(f'TP — обнаруженные атаки: {tp}')
    print(f'FP — ложные тревоги: {fp}')
    print(f'FN — пропущенные атаки: {fn}')
    print(f'TN — корректно распознанная норма: {tn}')

def calculate_knn_anomaly_score(distances, score_type='mean'):
    if score_type == 'mean':
        return distances.mean(axis=1)
    if score_type == 'kth':
        return distances[:, -1]
    raise ValueError("score_type должен быть 'mean' или 'kth'")

def scenario_table(prediction):
    result = monitoring_df.loc[monitoring_df['is_attack'] == 1, ['scenario']].copy()
    result['detected'] = np.asarray(prediction)[monitoring_df['is_attack'].to_numpy() == 1]
    return result.groupby('scenario')['detected'].agg(['sum', 'count'])

## Задание 1. Подготовка данных и режимов работы — 3 балла
Проведите аудит набора, подготовьте стандартизованную смешанную выборку и раздельные выборки `normal_fit`, `normal_calibration`, `monitoring`. Объясните, почему служебные метки не входят в признаки и почему scaler профиля нормы обучается только на `normal_fit`.

In [ ]:
# выполните аудит и подготовьте X_unsupervised,
# X_normal_fit, X_normal_calibration и X_monitoring.

## Задание 2. kNN в двух режимах — 6 баллов
Используйте `k=10`, евклидово расстояние и среднее расстояние до соседей. В смешанном режиме исключите сам объект вызовом `kneighbors()` без `X` и выберите порог по доле 0,055. Для профиля нормы выберите 99-й процентиль оценок калибровочной нормы. Выведите пороги, TP, FP, FN, TN и обнаружение по сценариям.

In [ ]:
# реализуйте kNN в двух режимах.
# Подсказка: средняя оценка = distances.mean(axis=1).

## Задание 3. LOF в двух режимах — 6 баллов
Используйте `n_neighbors=20`. В смешанном режиме задайте `contamination=0.055`, `novelty=False`, примените `fit_predict()` и смените знак `negative_outlier_factor_`. Для профиля нормы задайте `novelty=True`, получите положительные оценки через `-score_samples()` и выберите 99-й процентиль калибровочной нормы.

In [ ]:
# реализуйте LOF в двух режимах.
# Помните о разных источниках оценок и смене знака.

## Задание 4. Сравнение и интерпретация для SOC — 5 баллов
Сведите результаты четырёх комбинаций метода и режима в таблицу, постройте гистограммы оценок с порогами и исследуйте нормальных соседей наиболее аномальной по kNN сессии. Подготовьте аналитическую записку SOC из 5–7 предложений.

In [ ]:
# сформируйте итоговую таблицу, четыре гистограммы
# и выведите соседей наиболее подозрительной сессии.

### Аналитическая записка SOC
Объём и состав анализа …

Методы, режимы и пороги …

Результаты …

Приоритетные сессии и отличие от соседей …

Ограничения …

Рекомендуемые проверки …